# Same job, borrowed brain (transformers)

Your LSTM in `sentimentAnalysis.ipynb` got **88%**, and it learned English from scratch
using 35,000 movie reviews. That is like teaching someone the language *and* the job at
the same time.

A **transformer** is a different kind of reader, and the ones on Hugging Face come
already trained on enormous amounts of English.

**LSTM:** reads left to right, one word at a time, keeping a single running memory.
By word 180 it has mostly forgotten word 3.

**Transformer:** sees the whole review at once and works out which words matter to which.
In *"I expected it to be awful, but it turned out great"*, it can connect "but" to both
halves and understand the reversal. That exact sentence is the one your LSTM kept
getting wrong.

We will use **DistilBERT**, a smaller, faster version of BERT, in the two ways people
actually use these models:

- **Part A:** borrow one that is *already* trained to judge sentiment. No training at all.
- **Part B:** take the plain English-reading one and teach it your IMDB reviews
  (this is fine-tuning, the same idea as ResNet18 in `catDogTransfer.ipynb`).

In [ ]:
# Run this once. It installs the Hugging Face library into your venv.
%pip install transformers

In [ ]:
#step 1 - tools and data

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device-{device}")

dataframe = pd.read_csv('IMDB_Dataset.csv')

# exactly the same split as the LSTM notebook, so the scores are comparable
training_df = dataframe.sample(frac=0.7, random_state=42)
test_df = dataframe.drop(training_df.index)

print(f"{len(training_df)} training reviews, {len(test_df)} test reviews")

## Part A: borrow one that already does the job

Someone has already taken DistilBERT and trained it on a pile of sentiment examples.
It is sitting on Hugging Face under the name
`distilbert-base-uncased-finetuned-sst-2-english`.

`pipeline` is the easy button: it downloads the model, handles all the text preparation,
and gives back an answer in plain English. No dictionary, no padding code, no training.

The first run downloads about 250MB, then it is cached on your machine.

In [ ]:
#step 2 - the ready-made model, no training at all

from transformers import pipeline

reader = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device,
)

# the same four reviews that tripped up the LSTM
for review in [
    "This movie was wonderful. Brilliant acting and I loved every minute of it.",
    "What a waste of two hours. Boring plot, terrible acting, I want my money back.",
    "I expected it to be awful, but honestly it turned out to be great.",
    "It was not good.",
]:
    answer = reader(review)[0]                      # e.g. {'label': 'POSITIVE', 'score': 0.99}
    mood = answer["label"].lower()
    sureness = answer["score"] * 100
    print(f"{mood} ({sureness:.1f}% sure)  <-  {review}")

In [ ]:
#step 3 - grade the ready-made model on your test reviews

# 2000 reviews is plenty to get a reliable score, and keeps this quick
sample_df = test_df.head(2000)
reviews = sample_df["review"].tolist()
real_answers = sample_df["sentiment"].tolist()

# truncation=True: IMDB reviews can be longer than the model's 512-word limit, so cut them
guesses = reader(reviews, batch_size=32, truncation=True, max_length=512)

correct = 0
for guess, real in zip(guesses, real_answers):
    if guess["label"].lower() == real:
        correct += 1

print(f"ready-made model, zero training: {correct / len(reviews) * 100:.1f}%")
print(f"your LSTM, trained for 10 epochs: 88.0%")

## Part B: fine-tune one on your own reviews

Part A used someone else's sentiment model. Now we take **plain** DistilBERT — it knows
English but has never judged a review — and teach it the job on your IMDB data. This is
exactly what you did with ResNet18 for cats and dogs.

Two things replace a lot of your old code:

**The tokenizer replaces `clean_text`, the vocabulary and `encode`.** It already has its
own dictionary (the same one the model was trained with), and it does the cleaning,
the ID lookup and the padding in one call. It also hands back an `attention_mask`:
a list of 1s and 0s marking which slots are real words and which are blank filler, so
the model knows to ignore the padding.

**A tiny learning rate.** `2e-5` is 0.00002 — fifty times smaller than the `0.001` you
used before. The model already knows English, and big nudges would wreck that knowledge.
You are adjusting it, not teaching it from scratch.

One epoch is usually enough. We use 5,000 reviews here to keep it to a few minutes;
raise `HOW_MANY` if you want to push the score higher.

In [ ]:
#step 4 - prepare the text with the model's own tokenizer

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"    # plain DistilBERT: knows English, knows nothing about reviews
MAX_LEN = 256                             # how many words of each review to read
HOW_MANY = 5000                           # training reviews to use. 5000 takes ~4.5 min and scores ~88%

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_part = training_df.head(HOW_MANY)
test_part = test_df.head(2000)


def prepare(df):
    """Turn a table of reviews into number-blocks the model can read."""
    packed = tokenizer(
        df["review"].tolist(),
        truncation=True,          # cut reviews longer than MAX_LEN
        padding="max_length",     # pad short ones so every review is the same length
        max_length=MAX_LEN,
        return_tensors="pt",      # give it back as PyTorch tensors
    )
    labels = torch.tensor(df["sentiment"].map({"negative": 0, "positive": 1}).values)
    return TensorDataset(packed["input_ids"], packed["attention_mask"], labels)


train_dataloader = DataLoader(prepare(train_part), batch_size=16, shuffle=True)
test_dataloader = DataLoader(prepare(test_part), batch_size=32)

print(f"{len(train_part)} training reviews, {len(test_part)} test reviews, ready")


In [ ]:
#step 5 - fine-tune it (same loop you already know)

epochs = 1

# num_labels=2 bolts a fresh 2-way decision layer onto the end, same as swapping ResNet's fc
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

loss_fun = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)    # tiny steps: it already knows English


def check_score():
    """Grade the model on the test reviews, give back the % it got right."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for ids, mask, labels in test_dataloader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            guess = model(ids, attention_mask=mask).logits    # .logits holds the 2 raw scores
            answer = guess.argmax(dim=1)
            correct += (answer == labels).sum().item()
            total += len(labels)
    model.train()
    return correct / total * 100


model.train()
for epoch in range(epochs):
    total_loss = 0

    for ids, mask, labels in train_dataloader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        optimizer.zero_grad()                              # wipe the last batch's corrections
        guess = model(ids, attention_mask=mask).logits     # what does it think?
        loss = loss_fun(guess, labels)                     # how wrong was it?
        loss.backward()                                    # which way to nudge
        optimizer.step()                                   # nudge

        total_loss += loss.item()

    score = check_score()
    print(f"Epoch {epoch+1}: avg loss {total_loss / len(train_dataloader):.4f}  test {score:.1f}%")

model.save_pretrained("sentiment_distilbert")    # saves the model and its settings in a folder
tokenizer.save_pretrained("sentiment_distilbert")
print("saved to sentiment_distilbert/")

In [ ]:
#step 6 - try your fine-tuned model on your own reviews

def how_does_it_feel(review):
    """Say whether one review is positive or negative, and how sure the model is."""
    packed = tokenizer(review, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ids = packed["input_ids"].to(device)
    mask = packed["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        scores = model(ids, attention_mask=mask).logits
        chances = torch.softmax(scores, dim=1)[0]        # 2 scores -> 2 percentages

    if chances[1] > chances[0]:
        return "positive", chances[1].item() * 100
    return "negative", chances[0].item() * 100


for review in [
    "This movie was wonderful. Brilliant acting and I loved every minute of it.",
    "What a waste of two hours. Boring plot, terrible acting, I want my money back.",
    "I expected it to be awful, but honestly it turned out to be great.",
    "It was not good.",
]:
    mood, sureness = how_does_it_feel(review)
    print(f"{mood} ({sureness:.1f}% sure)  <-  {review}")

## What to notice

Measured on this machine (M-series Mac, `mps`), 2,000 test reviews:

| Approach | Training | Score |
|---|---|---|
| Your LSTM | 10 epochs on 35,000 reviews | 88.0% |
| **Part A**, ready-made model | none at all | **89.8%** |
| **Part B**, fine-tuned DistilBERT | 1 epoch on 5,000 reviews (4.5 min) | 88.4% |

**Part B did not beat Part A, and that is worth understanding.** 5,000 reviews and a
single pass is not much teaching. The Part A model was trained by someone else on a large
sentiment dataset, so it arrives already good at this exact job. Raise `HOW_MANY` to
20000 and Part B should pass it — each 4x more data costs roughly 4x the time.

**How much code disappeared.** No `clean_text`, no `Counter`, no vocabulary, no `encode`,
no padding logic. The tokenizer does all of it, and it uses the exact dictionary the model
was trained with — which matters, because a borrowed model needs its input prepared the
way it was during its own training.

**The scary-looking LOAD REPORT** that Part B prints is normal. It says
`classifier.weight MISSING`, which means the 2-way decision layer was created fresh with
random numbers — exactly what you want, because that is the part you are about to train.

**The price.** DistilBERT holds 66 million numbers against your LSTM's roughly 800,000,
so each pass is far slower and the saved model is about 250MB instead of 3MB. For a job
this simple, the LSTM is not a bad answer — it is a fair trade of accuracy for size.

**Things to try next**

- Raise `HOW_MANY` to 20000 and watch the score climb.
- Swap `MODEL_NAME` for `"bert-base-uncased"` (bigger, slower, a bit better) or
  `"roberta-base"`.
- Freeze the body and train only the head, the way you froze ResNet18:
  `for p in model.distilbert.parameters(): p.requires_grad = False`. Much faster,
  a few points less accurate.
